# Run one corrected v2 experiment
GPU required. The actual experiment logic lives in one tested script, so this notebook only chooses the run. It records stage-0 validation, uses the audited per-method replay budget, saves pair-level margins, and marks a run complete only after all stages finish.

Run the core grid first: 2 orders × 3 methods (`none`, `random`, `mfr`) × 2 seeds = 12 runs. Then run the four `random_high` controls (18 new + 3 random replay pairs). Treat `lowest_margin` as an optional secondary baseline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import os, sys, subprocess
if not os.path.exists('/content/mfr-dpo'):
    !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
elif subprocess.run(['git', '-C', '/content/mfr-dpo', 'status', '--porcelain', '--', 'data/v2'], capture_output=True, text=True, check=True).stdout.strip():
    print('Keeping local data/v2; skipping git pull so it is not overwritten.')
else:
    !git -C /content/mfr-dpo pull --ff-only -q
!pip install -q -r /content/mfr-dpo/requirements.txt
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/mfr-dpo'
DRIVE_DIR = '/content/drive/MyDrive/CSCI544/mfr-dpo'

Mounted at /content/drive


Change only the values below. For replay methods, `STAGE1_FROM` should point to the matching completed v2 `none` run directory for the same order and seed. Never point it to the old pilot.

In [2]:
ORDER_ID = 1                # 1 or 2
METHOD = 'none'             # none | random | random_high | mfr | lowest_margin
SEED = 0                    # 0 or 1
START_STAGE = 1             # set 2 or 3 only when resuming this same run
STAGE1_FROM = None          # e.g. f'{DRIVE_DIR}/runs/v2_o2_none_s0'
REFERENCE_CACHE = f'{DRIVE_DIR}/cache/reference_v2.csv'

In [3]:
command = [sys.executable, '-u', f'{REPO}/scripts/run_experiment.py', '--drive-dir', DRIVE_DIR,
           '--order', str(ORDER_ID), '--method', METHOD, '--seed', str(SEED),
           '--start-stage', str(START_STAGE)]
if STAGE1_FROM:
    command += ['--stage1-from', STAGE1_FROM]
if os.path.exists(REFERENCE_CACHE):
    command += ['--reference-cache', REFERENCE_CACHE]
print(' '.join(command))
subprocess.run(command, check=True)

/usr/bin/python3 -u /content/mfr-dpo/scripts/run_experiment.py --drive-dir /content/drive/MyDrive/CSCI544/mfr-dpo --order 1 --method none --seed 0 --start-stage 1 --reference-cache /content/drive/MyDrive/CSCI544/mfr-dpo/cache/reference_v2.csv


CalledProcessError: Command '['/usr/bin/python3', '-u', '/content/mfr-dpo/scripts/run_experiment.py', '--drive-dir', '/content/drive/MyDrive/CSCI544/mfr-dpo', '--order', '1', '--method', 'none', '--seed', '0', '--start-stage', '1', '--reference-cache', '/content/drive/MyDrive/CSCI544/mfr-dpo/cache/reference_v2.csv']' returned non-zero exit status 1.

A finished run ends with `Complete:` and contains `COMPLETE.json`. If Colab disconnects, rerun with `START_STAGE` set to the first unfinished stage. Do not delete completed artifacts.